In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
import pandas as pd
temp_df = pd.read_csv('../data/AAPL_data.csv', nrows=0)
print(temp_df.columns.tolist())


['Price', 'Close', 'High', 'Low', 'Open', 'Volume']


In [7]:
df = pd.read_csv("../data/AAPL_data.csv", index_col="Price", parse_dates=True)
print(df.shape)
print(df.head())

(2518, 5)
                         Close                High                 Low  \
Price                                                                    
Ticker                    AAPL                AAPL                AAPL   
Date                       NaN                 NaN                 NaN   
2015-01-02   24.19260025024414  24.659502293956603  23.754464067072234   
2015-01-05  23.511062622070312  24.042136374239757  23.325187737341313   
2015-01-06  23.513269424438477    23.7721672375332   23.15258108261458   

                          Open     Volume  
Price                                      
Ticker                    AAPL       AAPL  
Date                       NaN        NaN  
2015-01-02  24.648437592390618  212818400  
2015-01-05  23.962475227002493  257142000  
2015-01-06  23.575227702283048  263188400  


C:\Users\Welcome\AppData\Local\Temp\ipykernel_29092\3526746204.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv("../data/AAPL_data.csv", index_col="Price", parse_dates=True)


In [ ]:
print(df.iloc[2].apply(type))   #numerical numbers are read as strings

Close     <class 'numpy.float64'>
High                <class 'str'>
Low                 <class 'str'>
Open                <class 'str'>
Volume              <class 'str'>
return    <class 'numpy.float64'>
Name: 2015-01-02, dtype: object


adding a new column as "return" and pct_change() calculates how much the price moved in percentage terms each day. If the stock went from $100 to $102, the return is 0.02 (2%). This is the core signal everything else is built from.

In [11]:
df["Close"] = pd.to_numeric(df["Close"].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

In [12]:
df["return"] = df["Close"].pct_change()

In [13]:
print(df.head())

                Close                High                 Low  \
Price                                                           
Ticker            NaN                AAPL                AAPL   
Date              NaN                 NaN                 NaN   
2015-01-02  24.192600  24.659502293956603  23.754464067072234   
2015-01-05  23.511063  24.042136374239757  23.325187737341313   
2015-01-06  23.513269    23.7721672375332   23.15258108261458   

                          Open     Volume    return  
Price                                                
Ticker                    AAPL       AAPL       NaN  
Date                       NaN        NaN       NaN  
2015-01-02  24.648437592390618  212818400       NaN  
2015-01-05  23.962475227002493  257142000 -0.028171  
2015-01-06  23.575227702283048  263188400  0.000094  


Financial markets often display short-term patterns that lag variables expose: <br>
**Momentum:** If lag1 and lag2 are highly positive, the asset might be in a strong upward trend.<br>
**Mean Reversion:** If an asset went up too high over lag1, lag2, and lag3, it might be due for a downward correction.<br><br>
shift(1) moves the column down by one row — so today's row now contains yesterday's value. You're giving the model a short memory of recent price movements.

In [ ]:
df["lag1"] = df["return"].shift(1)  # yesterday's return
df["lag2"] = df["return"].shift(2)  # two days ago
df["lag3"] = df["return"].shift(3)  # three days ago

In [ ]:
df["sma5"]  = df["Close"].rolling(5).mean() # Average of the last 5 days
df["sma20"] = df["Close"].rolling(20).mean() # Average of the last 20 days
df["sma50"] = df["Close"].rolling(50).mean() # Average of the last 50 days

#### Price relative to moving average
| Result | What it means |
|---|---|
| **= 1.0** | Stock is trading *exactly* at its average — perfectly normal |
| **> 1.0** (e.g. 1.15) | Stock is trading *above* its average — it's been running hot |
| **< 1.0** (e.g. 0.85) | Stock is trading *below* its average — it's been cooling off |

This code is comparing today's stock price to the moving averages you calculated before. It's asking:

In [ ]:
df["price_to_sma20"] = df["Close"] / df["sma20"]
df["price_to_sma50"] = df["Close"] / df["sma50"]

Standard deviation of returns over the last 10 or 20 days. High volatility means the stock has been jumping around a lot lately. Low volatility means it's been calm. The model can use this to adjust its confidence.

In [ ]:
df["volatility_10"] = df["return"].rolling(10).std()
df["volatility_20"] = df["return"].rolling(20).std()